# DAG enrichment scaling experiment

Tests both modes across a range of corruption levels on single-ordering ProScript plans.

**Three experiments run in sequence:**

1. **Mode 1 deletion sweep** — remove 20 / 40 / 60% of load-bearing edges; probe recovers them
2. **Mode 2 spurious insertion** — add 1 / 2 / 3 spurious shortcut edges per plan; probe removes them
3. **Combined** — 40% deletion + 2 spurious shortcuts; both modes run on the same corrupted DAG

Scaling curves (F1 vs. corruption level) are the primary output.

**Required files in `/content/`:**

| File | Role |
|---|---|
| `proScript_data-20260226T065943Z-1-001.zip` | All plan JSONs |
| `proscript_train_edges_v2.csv` | Probe training (confirmed + reversed) |
| `proscript_train_edges_v3.csv` | Multi-ordering plan identification |
| `proscript_pipeline_eval_final.csv` | Eval-set goals — excluded from training |

If `cfg['retrain'] = False`, also upload:
- `probe1_tp1_ordering_layer17.pkl`
- `probe2_tp1_layer17.pkl`
(generated by `validation_notebook.ipynb`)


In [15]:
import subprocess, zipfile, os
subprocess.run(['pip','install','-q','transformers','accelerate','scikit-learn','tqdm','matplotlib'],check=True)
BASE_DIR='/content'; DATA_DIR='/content/proScript_data'
os.makedirs(DATA_DIR, exist_ok=True)
zip_c=[f for f in os.listdir(BASE_DIR) if 'proScript_data' in f and f.endswith('.zip')]
if len([f for f in os.listdir(DATA_DIR) if f.endswith('.json')])<600:
    assert zip_c
    with zipfile.ZipFile(os.path.join(BASE_DIR,zip_c[0])) as z: z.extractall(BASE_DIR)
n_json=len([f for f in os.listdir(DATA_DIR) if f.endswith('.json')])
assert n_json>=600, f'Expected ~622 JSON files, found {n_json}'
required=['proscript_train_edges_v2.csv','proscript_train_edges_v3.csv',
          'proscript_pipeline_eval_final.csv']
missing=[f for f in required if not os.path.exists(os.path.join(BASE_DIR,f))]
assert not missing, f'Missing: {missing}'
print(f'✓ {n_json} JSON files  ✓ all CSVs present')


In [16]:
import json as _j, random, warnings, math
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')
CONTENT=Path('/content')

cfg={
    'model_name':    'mistralai/Mistral-7B-Instruct-v0.1',
    'probe_layer':   17,
    'retrain':       True,
    # Mode 1: fraction of load-bearing edges removed
    'deletion_rates':   [0.20, 0.40, 0.60],
    # Mode 2: shortcut edges added per plan
    'spurious_levels':  [1, 2, 3],
    # Combined experiment
    'combined_deletion': 0.40,
    'combined_spurious': 2,
    # Evaluation
    'thresholds': [0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70],
    'seed':  42,
    'chat_template': True,
}
SEED=cfg['seed']
print('Config ready')


In [17]:
import json as _json, os, re
from collections import defaultdict, deque

def parse_plan_json(goal, data_dir):
    path=os.path.join(data_dir,goal.replace(' ','_')+'.json')
    if not os.path.exists(path): return None
    with open(path,encoding='utf-8-sig') as f: raw=f.read().strip()
    if not raw: return None
    d=_json.loads(raw)
    steps={str(k):v for k,v in d['steps'].items()}
    step_map,start_n,end_n={},None,None
    for k,v in steps.items():
        ki,vs=int(k),v.strip()
        if vs.upper()=='START': start_n=ki
        elif vs.upper()=='END':  end_n=ki
        else:                    step_map[ki]=vs
    adj=defaultdict(list); dag_edges=[]
    for a_s,b_s in [(str(a),str(b)) for a,b in d['edges']]:
        a,b=int(a_s),int(b_s)
        if start_n is not None and a==start_n: continue
        if end_n   is not None and b==end_n:   continue
        if a in step_map and b in step_map:
            adj[a].append(b); dag_edges.append((step_map[a],step_map[b]))
    real=list(step_map.keys())
    def reach(s):
        v,q=set(),[s]
        while q:
            n=q.pop()
            for nb in adj.get(n,[]):
                if nb not in v: v.add(nb); q.append(nb)
        return v
    r={n:reach(n) for n in real}
    incompat=[(step_map[n1],step_map[n2]) for i,n1 in enumerate(real)
              for n2 in real[i+1:] if n2 not in r[n1] and n1 not in r[n2]]
    return {'steps':step_map,'dag_edges':dag_edges,'incomparable':incompat}

def compute_step_depths(steps_list, edges_list):
    adj=defaultdict(set); in_deg={s:0 for s in steps_list}
    for a,b in edges_list: adj[a].add(b); in_deg[b]=in_deg.get(b,0)+1
    depth={s:0 for s in steps_list}
    q=deque([s for s in steps_list if in_deg.get(s,0)==0])
    while q:
        n=q.popleft()
        for nb in adj.get(n,set()):
            depth[nb]=max(depth[nb],depth[n]+1)
            in_deg[nb]-=1
            if in_deg[nb]==0: q.append(nb)
    return depth

def has_alternative_path(edges, a, b):
    """True if b is reachable from a without the direct a→b edge."""
    adj=defaultdict(set)
    for x,y in edges:
        if not (x==a and y==b): adj[x].add(y)
    v,q=set(),[a]
    while q:
        n=q.pop()
        if n==b: return True
        for nb in adj.get(n,set()):
            if nb not in v: v.add(nb); q.append(nb)
    return False

def get_incompat_pairs(steps_list, edges_list):
    adj=defaultdict(set)
    for a,b in edges_list: adj[a].add(b)
    def r(s):
        v,q=set(),[s]
        while q:
            n=q.pop()
            for nb in adj.get(n,set()):
                if nb not in v: v.add(nb); q.append(nb)
        return v
    reach={s:r(s) for s in steps_list}
    return {(a,b) for a in steps_list for b in steps_list
            if a!=b and b not in reach[a] and a not in reach[b]}

def find_spurious_same_depth(plan):
    """
    Mode 2a: spurious edges between genuinely parallel steps.
    Adds A→B where A and B are incomparable AND at the same topological depth.
    These are the closest match to Probe 2's training flexible class.
    Expected: best probe performance of the three modes.
    """
    all_steps = list(plan['steps'].values())
    depths    = compute_step_depths(all_steps, plan['dag_edges'])
    incompat  = get_incompat_pairs(all_steps, plan['dag_edges'])
    return [(a,b) for a,b in incompat
            if depths.get(a) == depths.get(b)]

def find_spurious_cross_branch(plan):
    """
    Mode 2b: spurious edges between steps in independent branches.
    Adds A→B where A and B are incomparable AND at different depths.
    Tests whether the probe can reject constraints across independent sub-procedures.
    Expected: medium probe performance.
    """
    all_steps = list(plan['steps'].values())
    depths    = compute_step_depths(all_steps, plan['dag_edges'])
    incompat  = get_incompat_pairs(all_steps, plan['dag_edges'])
    return [(a,b) for a,b in incompat
            if depths.get(a) != depths.get(b)]

def find_transitive_shortcuts(dag_edges):
    """
    Returns all (A,C) pairs where A→B→C exists in the DAG but A→C is not a
    direct edge. These are the most natural spurious edges: redundant but
    semantically plausible. Mode 2's job is to identify and remove them.
    """
    edge_set=set(dag_edges)
    adj=defaultdict(set)
    for a,b in dag_edges: adj[a].add(b)
    shortcuts=[]
    for a in list(adj.keys()):
        for b in adj[a]:
            for c in adj[b]:
                if (a,c) not in edge_set and a!=c:
                    shortcuts.append((a,c))
    return list(set(shortcuts))

print('Utils ready')


In [18]:
def wrap(raw): return f'[INST] {raw.strip()} [/INST]' if cfg['chat_template'] else raw

def tp1_ordering(goal,a,b):
    return wrap(f'You are judging a temporal dependency between two actions in a task.\n'
                f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
                f'Question: Must Action A happen before Action B? Answer yes or no.\nAnswer:')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
print(f'Loading {cfg["model_name"]} ...')
tokenizer=AutoTokenizer.from_pretrained(cfg['model_name'])
model=AutoModelForCausalLM.from_pretrained(
    cfg['model_name'],torch_dtype=torch.float16,
    device_map='auto',output_hidden_states=True)
model.eval()
YES_IDS=[tokenizer.encode(t,add_special_tokens=False)[0] for t in (' yes','yes','Yes')]
NO_IDS =[tokenizer.encode(t,add_special_tokens=False)[0] for t in (' no', 'no', 'No')]

@torch.no_grad()
def get_hidden_and_pyes(prompt,layer):
    inputs=tokenizer(prompt,return_tensors='pt').to(model.device)
    out=model(**inputs,output_hidden_states=True)
    h=out.hidden_states[layer][0,-1].float().cpu().numpy()
    probs=torch.softmax(out.logits[0,-1].float(),dim=-1).cpu()
    p_yes=sum(probs[i].item() for i in YES_IDS)
    p_no =sum(probs[i].item() for i in NO_IDS)
    denom=p_yes+p_no if (p_yes+p_no)>0 else 1.0
    return h, p_yes/denom

print('Model and feature helpers ready')


In [19]:
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

base_v2 =pd.read_csv(CONTENT/'proscript_train_edges_v2.csv')
base_v3 =pd.read_csv(CONTENT/'proscript_train_edges_v3.csv')
ps_eval =pd.read_csv(CONTENT/'proscript_pipeline_eval_final.csv')
eval_goals=set(ps_eval['goal'])
multi_goals=set(base_v3[base_v3.edge_type=='truly_parallel']['goal'])-eval_goals

# ── Identify test plans ───────────────────────────────────────────────────────
# Group C single-ordering plans (not in eval label, just not multi-ordering)
all_gc_plans=[]
for goal in eval_goals:
    plan=parse_plan_json(goal,DATA_DIR)
    if plan is None: continue
    all_gc_plans.append({'goal':goal,'is_multi':len(plan['incomparable'])>0})
gc_df=pd.DataFrame(all_gc_plans)
single_plans=set(gc_df[~gc_df.is_multi]['goal'])
multi_plans_gc=set(gc_df[gc_df.is_multi]['goal'])
print(f'Group C: {len(single_plans)} single-ordering, {len(multi_plans_gc)} multi-ordering')

# ── Load or train probes ──────────────────────────────────────────────────────
if not cfg['retrain']:
    p1_path=CONTENT/f'probe1_tp1_ordering_layer{cfg["probe_layer"]}.pkl'
    p2_path=CONTENT/f'probe2_tp1_layer{cfg["probe_layer"]}.pkl'
    assert p1_path.exists() and p2_path.exists(), \
        'Probe pkl files not found. Run validation_notebook.ipynb first or set retrain=True'
    with open(p1_path,'rb') as f: probe1=pickle.load(f)
    with open(p2_path,'rb') as f: probe2=pickle.load(f)
    print('Probes loaded from file')
else:
    # ── Probe 1: confirmed+reversed vs incomparable (binary) ─────────────────
    confirmed =base_v2[(base_v2.label==1)&(~base_v2.goal.isin(eval_goals))].copy()
    reversed_ =base_v2[(base_v2.label==0)&(~base_v2.goal.isin(eval_goals))].copy()
    incompat_rows=[]
    for goal in sorted(multi_goals):
        plan=parse_plan_json(goal,DATA_DIR)
        if not plan: continue
        for a,b in plan['incomparable']:
            incompat_rows+=[{'goal':goal,'a':a,'b':b,'label':0},
                             {'goal':goal,'a':b,'b':a,'label':0}]
    neg_pool=pd.concat([reversed_,pd.DataFrame(incompat_rows)],ignore_index=True)
    n1=min(len(confirmed),len(neg_pool))
    p1_train=pd.concat([
        confirmed.sample(n1,random_state=SEED).assign(label=1),
        neg_pool.sample(n1,random_state=SEED).assign(label=0),
    ],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
    print(f'Probe 1 training: {len(p1_train)} rows ({n1}/class)')
    X1,y1=[],[]
    for _,row in tqdm(p1_train.iterrows(),total=len(p1_train),desc='Probe 1 features'):
        h,_=get_hidden_and_pyes(tp1_ordering(row.goal,row.a,row.b),cfg['probe_layer'])
        X1.append(h); y1.append(row.label)
    X1,y1=np.array(X1),np.array(y1)
    probe1=LogisticRegression(C=1.0,max_iter=1000,class_weight='balanced').fit(X1,y1)
    with open(CONTENT/f'probe1_tp1_ordering_layer{cfg["probe_layer"]}.pkl','wb') as f:
        pickle.dump(probe1,f)
    # ── Probe 2: confirmed_keep vs same-depth spurious (binary) ──────────────
    conf_keep=base_v2[(base_v2.label==1)&(~base_v2.goal.isin(eval_goals))].copy()
    spur_rows=[]
    for goal in sorted(multi_goals):
        plan=parse_plan_json(goal,DATA_DIR)
        if not plan: continue
        steps_list=list(plan['steps'].values())
        depths=compute_step_depths(steps_list,plan['dag_edges'])
        for a,b in plan['incomparable']:
            if depths.get(a)==depths.get(b):
                spur_rows+=[{'goal':goal,'a':a,'b':b,'probe_label':0},
                             {'goal':goal,'a':b,'b':a,'probe_label':0}]
    spurious_df=pd.DataFrame(spur_rows)
    n2=min(len(conf_keep),len(spurious_df))
    p2_train=pd.concat([
        conf_keep.sample(n2,random_state=SEED).assign(probe_label=1),
        spurious_df.sample(n2,random_state=SEED).assign(probe_label=0),
    ],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
    print(f'Probe 2 training: {len(p2_train)} rows ({n2}/class)')
    X2,y2=[],[]
    for _,row in tqdm(p2_train.iterrows(),total=len(p2_train),desc='Probe 2 features'):
        h,_=get_hidden_and_pyes(tp1_ordering(row.goal,row.a,row.b),cfg['probe_layer'])
        X2.append(h); y2.append(row.probe_label)
    X2,y2=np.array(X2),np.array(y2)
    probe2=LogisticRegression(C=1.0,max_iter=1000,class_weight='balanced').fit(X2,y2)
    with open(CONTENT/f'probe2_tp1_layer{cfg["probe_layer"]}.pkl','wb') as f:
        pickle.dump(probe2,f)
    print('Both probes trained and saved')

def p1_score(h): return probe1.predict_proba(h.reshape(1,-1))[0,1]
def p2_spurious(h): return probe2.predict_proba(h.reshape(1,-1))[0,0]
print('Probes ready')


In [20]:
def clf_metrics(y_true, y_pred):
    yt,yp=np.array(y_true),np.array(y_pred)
    tp=int(((yp==1)&(yt==1)).sum()); fp=int(((yp==1)&(yt==0)).sum())
    tn=int(((yp==0)&(yt==0)).sum()); fn=int(((yp==0)&(yt==1)).sum())
    prec=tp/(tp+fp) if (tp+fp) else 0.0
    rec =tp/(tp+fn) if (tp+fn) else 0.0
    f1  =2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    return dict(tp=tp,fp=fp,tn=tn,fn=fn,
                precision=round(prec,4),recall=round(rec,4),f1=round(f1,4))

def sweep_probe(eval_df, score_fn, label):
    """score_fn(h) → float; higher = more likely positive."""
    scores=[]
    for _,r in tqdm(eval_df.iterrows(),total=len(eval_df),desc=label):
        h,_=get_hidden_and_pyes(tp1_ordering(r.goal,r.a,r.b),cfg['probe_layer'])
        scores.append(score_fn(h))
    rows=[]
    for t in cfg['thresholds']:
        preds=[int(s>t) for s in scores]
        m=clf_metrics(eval_df.y_true.tolist(),preds)
        rows.append({'threshold':t,**m})
    return pd.DataFrame(rows), scores

def sweep_llm_m1(eval_df, label):
    """LLM direct: p_yes > t → predict ordered pair."""
    scores=[]
    for _,r in tqdm(eval_df.iterrows(),total=len(eval_df),desc=f'LLM {label}'):
        _,p_yes=get_hidden_and_pyes(tp1_ordering(r.goal,r.a,r.b),cfg['probe_layer'])
        scores.append(p_yes)
    rows=[]
    for t in cfg['thresholds']:
        preds=[int(s>t) for s in scores]
        m=clf_metrics(eval_df.y_true.tolist(),preds)
        rows.append({'threshold':t,**m})
    return pd.DataFrame(rows), scores

def best_f1(sweep_df): return sweep_df.loc[sweep_df.f1.idxmax()]

print('Eval helpers ready')


---
## Mode 1 — deletion sweep

For each deletion rate (20 / 40 / 60%), load-bearing edges are removed from
single-ordering Group C plans and the probe must identify them as ordered pairs.

**Negatives** come from multi-ordering Group C plans — genuinely flexible pairs
the probe should correctly abstain on.

**Caveats at 60% deletion:**
some plans run out of hard edges (a 5-step chain may have only 4 hard edges);
those plans contribute fewer positives than requested.
Beyond 60%, distribution shift from training dominates performance — excluded.


In [21]:
rng_m1=random.Random(SEED)

# Negatives: incomparable pairs from multi-ordering Group C plans
neg_rows=[]
for goal in sorted(multi_plans_gc):
    plan=parse_plan_json(goal,DATA_DIR)
    if not plan: continue
    for a,b in plan['incomparable']:
        neg_rows+=[{'goal':goal,'a':a,'b':b,'y_true':0},
                   {'goal':goal,'a':b,'b':a,'y_true':0}]
all_negs=pd.DataFrame(neg_rows)

# Positives: removed hard edges at each deletion rate
m1_sets={} # rate -> balanced DataFrame
for rate in cfg['deletion_rates']:
    pos_rows=[]
    for goal in sorted(single_plans):
        plan=parse_plan_json(goal,DATA_DIR)
        if not plan: continue
        hard=[(a,b) for a,b in plan['dag_edges']
              if not has_alternative_path(plan['dag_edges'],a,b)]
        if not hard: continue
        n_rem=max(1,math.ceil(rate*len(hard)))
        for a,b in rng_m1.sample(hard,min(n_rem,len(hard))):
            pos_rows.append({'goal':goal,'a':a,'b':b,'y_true':1})
    pos_df=pd.DataFrame(pos_rows)
    n=min(len(pos_df),len(all_negs))
    balanced=pd.concat([
        pos_df.sample(n,random_state=SEED),
        all_negs.sample(n,random_state=SEED),
    ],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
    m1_sets[rate]=balanced
    print(f'Mode 1 {int(rate*100)}%: {len(balanced)} rows  '
          f'pos={n}  from {pos_df.goal.nunique()} plans')


In [22]:
m1_results={} # rate -> {'probe':df, 'llm':df}
for rate in cfg['deletion_rates']:
    label=f'{int(rate*100)}pct'
    probe_df,_=sweep_probe(m1_sets[rate], p1_score,        f'Probe1 {label}')
    llm_df,  _=sweep_llm_m1(m1_sets[rate],                 f'LLM {label}')
    m1_results[rate]={'probe':probe_df,'llm':llm_df}

print('\n=== MODE 1 BEST F1 PER DELETION RATE ===')
for rate in cfg['deletion_rates']:
    pb=best_f1(m1_results[rate]['probe'])
    lb=best_f1(m1_results[rate]['llm'])
    margin=pb.f1-lb.f1
    print(f'  {int(rate*100)}%: Probe1 F1={pb.f1:.3f} (t={pb.threshold})  '
          f'LLM F1={lb.f1:.3f}  margin=+{margin:.3f}')


In [23]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,2,figsize=(12,4))

# Left: F1 scaling curve
ax=axes[0]
rates=[int(r*100) for r in cfg['deletion_rates']]
probe_f1s=[best_f1(m1_results[r]['probe']).f1 for r in cfg['deletion_rates']]
llm_f1s  =[best_f1(m1_results[r]['llm']).f1   for r in cfg['deletion_rates']]
ax.plot(rates,probe_f1s,'o-',color='#3C3489',lw=2,ms=7,label='Probe 1')
ax.plot(rates,llm_f1s,  's--',color='#888780',lw=1.5,ms=6,label='LLM direct')
ax.set_xlabel('Edges deleted (%)')
ax.set_ylabel('F1 (best threshold)')
ax.set_title('Mode 1: recovery F1 vs. deletion rate')
ax.set_xticks(rates); ax.legend(); ax.set_ylim(0,1)

# Right: precision / recall at best threshold for each rate
ax2=axes[1]
probe_prec=[best_f1(m1_results[r]['probe']).precision for r in cfg['deletion_rates']]
probe_rec =[best_f1(m1_results[r]['probe']).recall    for r in cfg['deletion_rates']]
ax2.plot(rates,probe_prec,'o-',color='#185FA5',lw=2,ms=7,label='precision')
ax2.plot(rates,probe_rec, 's--',color='#1D9E75',lw=2,ms=7,label='recall')
ax2.set_xlabel('Edges deleted (%)')
ax2.set_ylabel('Score'); ax2.set_title('Mode 1: precision & recall vs. deletion rate')
ax2.set_xticks(rates); ax2.legend(); ax2.set_ylim(0,1)

plt.tight_layout()
plt.savefig(CONTENT/'mode1_scaling.png',dpi=130,bbox_inches='tight')
plt.show(); print('Saved mode1_scaling.png')


---
## Mode 2 — spurious edge insertion (redesigned)

**Two spurious-edge subconditions, both using multi-ordering Group C plans:**

| Mode | Edge added | Training match | Expected performance |
|---|---|---|---|
| **2a same-depth** | Incomparable pair at same topological depth | ✓ Direct — matches training flexible class | Best |
| **2b cross-branch** | Incomparable pair at different depths | ≈ Close — same population, different depth | Strong |

Transitive shortcuts (A→B→C + A→C) are excluded — they test redundancy detection, not
parallelism rejection, and are outside the probe's training distribution.

**Negatives** = real existing edges from the same multi-ordering plans (should be kept).

**No leakage:** Probe 2 trained on Group B; these evaluations use Group C plans.


In [24]:
rng_m2 = random.Random(SEED + 1)

# Source: multi-ordering Group C plans (have genuine incomparable pairs)
# For each plan:
#   positives (y=1) = selected incomparable pairs added as spurious directed edges
#   negatives (y=0) = real existing edges from the same plan (should be kept)
# No transitive shortcuts — test distribution now matches training distribution.

SPURIOUS_FNS = {
    '2a_same_depth':   lambda plan: find_spurious_same_depth(plan),
    '2b_cross_branch': lambda plan: find_spurious_cross_branch(plan),
}

m2_sets = {}  # (mode_tag, n_spurious) -> balanced DataFrame

for mode_tag, spur_fn in SPURIOUS_FNS.items():
    for n_spur in cfg['spurious_levels']:
        pos_rows = []
        neg_rows = []
        for goal in sorted(multi_plans_gc):
            plan = parse_plan_json(goal, DATA_DIR)
            if not plan: continue
            candidates = spur_fn(plan)
            if not candidates: continue
            added = rng_m2.sample(candidates, min(n_spur, len(candidates)))
            for a, b in added:
                pos_rows.append({'goal':goal,'a':a,'b':b,'y_true':1})
            for a, b in plan['dag_edges']:
                neg_rows.append({'goal':goal,'a':a,'b':b,'y_true':0})
        if not pos_rows:
            print(f'  WARNING: {mode_tag} n={n_spur} — no candidates found')
            continue
        pos_df = pd.DataFrame(pos_rows)
        neg_df = pd.DataFrame(neg_rows)
        n = min(len(pos_df), len(neg_df))
        balanced = pd.concat([
            pos_df.sample(n, random_state=SEED),
            neg_df.sample(n, random_state=SEED),
        ], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
        m2_sets[(mode_tag, n_spur)] = balanced
        print(f'  Mode {mode_tag} n={n_spur}: {len(balanced)} rows  '
              f'pos={n} (from {pos_df.goal.nunique()} plans)  '
              f'neg={n} (real edges from same plans)')


In [25]:
def sweep_llm_m2(eval_df, label):
    """LLM low-confidence: p_yes < t → propose removal (spurious edge)."""
    scores = []
    for _,r in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f'LLM-inv {label}'):
        _,p_yes = get_hidden_and_pyes(tp1_ordering(r.goal,r.a,r.b), cfg['probe_layer'])
        scores.append(p_yes)
    rows = []
    for t in cfg['thresholds']:
        preds = [int(s < t) for s in scores]
        m = clf_metrics(eval_df.y_true.tolist(), preds)
        rows.append({'threshold':t,**m})
    return pd.DataFrame(rows)

m2_results = {}  # (mode_tag, n_spur) -> {'probe':df, 'llm':df}

for (mode_tag, n_spur), eval_df in m2_sets.items():
    label = f'{mode_tag}_n{n_spur}'
    probe_df,_ = sweep_probe(eval_df, p2_spurious, f'Probe2 {label}')
    llm_df     = sweep_llm_m2(eval_df,               f'LLM {label}')
    m2_results[(mode_tag, n_spur)] = {'probe':probe_df, 'llm':llm_df}

print('\n=== MODE 2 BEST F1 PER SUBCONDITION AND LEVEL ===')
for mode_tag in ['2a_same_depth','2b_cross_branch']:
    print(f'\n  {mode_tag}:')
    for n_spur in cfg['spurious_levels']:
        key = (mode_tag, n_spur)
        if key not in m2_results: continue
        pb = best_f1(m2_results[key]['probe'])
        lb = best_f1(m2_results[key]['llm'])
        winner = 'Probe' if pb.f1 >= lb.f1 else 'LLM-inv'
        print(f'    n={n_spur}: Probe={pb.f1:.3f} (t={pb.threshold})  '
              f'LLM-inv={lb.f1:.3f}  winner={winner}  '
              f'margin={pb.f1-lb.f1:+.3f}')


In [26]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

mode_colors = {
    '2a_same_depth':   '#3C3489',
    '2b_cross_branch': '#185FA5',
}
mode_labels = {
    '2a_same_depth':   '2a: same-depth parallel',
    '2b_cross_branch': '2b: cross-branch',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: F1 scaling curves — one line per mode, probe vs LLM
ax = axes[0]
levels = cfg['spurious_levels']
for mode_tag, color in mode_colors.items():
    probe_f1s = [best_f1(m2_results[(mode_tag,n)]['probe']).f1
                 for n in levels if (mode_tag,n) in m2_results]
    llm_f1s   = [best_f1(m2_results[(mode_tag,n)]['llm']).f1
                 for n in levels if (mode_tag,n) in m2_results]
    ax.plot(levels[:len(probe_f1s)], probe_f1s, 'o-',
            color=color, lw=2, ms=7, label=mode_labels[mode_tag])
    ax.plot(levels[:len(llm_f1s)], llm_f1s, 's--',
            color=color, lw=1, ms=5, alpha=0.5)
ax.set_xlabel('Spurious edges added per plan')
ax.set_ylabel('F1 (best threshold)')
ax.set_title('Mode 2: removal F1 by edge type\n(solid=Probe 2, dashed=LLM low-confidence)')
ax.set_xticks(levels); ax.legend(fontsize=9); ax.set_ylim(0, 1)

# Right: precision vs recall at best threshold, n=2, all modes
ax2 = axes[1]
n_plot = 2  # show n=2 for the comparison
for mode_tag, color in mode_colors.items():
    key = (mode_tag, n_plot)
    if key not in m2_results: continue
    pb = best_f1(m2_results[key]['probe'])
    lb = best_f1(m2_results[key]['llm'])
    ax2.scatter(pb.recall, pb.precision, color=color, s=120,
                zorder=5, label=f'{mode_labels[mode_tag]} (probe)')
    ax2.scatter(lb.recall, lb.precision, color=color, s=60,
                marker='s', zorder=5, alpha=0.5)
ax2.plot([0,1],[0,1],'--',color='#cccccc',lw=1)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title(f'Mode 2: precision vs recall at n={n_plot}\n(circles=probe, squares=LLM)')
ax2.legend(fontsize=8); ax2.set_xlim(0,1); ax2.set_ylim(0,1)

plt.tight_layout()
plt.savefig(CONTENT/'mode2_scaling.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved mode2_scaling.png')

# Summary interpretation
print('\n=== INTERPRETATION ===')
n_check = 2
outcomes = {}
for mode_tag in SPURIOUS_FNS:
    key = (mode_tag, n_check)
    if key not in m2_results: continue
    pb = best_f1(m2_results[key]['probe'])
    lb = best_f1(m2_results[key]['llm'])
    outcomes[mode_tag] = pb.f1 >= lb.f1
    print(f'  {mode_tag}: probe {"BEATS" if outcomes[mode_tag] else "LOSES TO"} LLM  '
          f'(probe={pb.f1:.3f} llm={lb.f1:.3f})')
print()
if outcomes.get('2a_same_depth') and outcomes.get('2b_cross_branch'):
    print('→ Probe correctly rejects both parallel and cross-branch spurious edges')
    print('  Layers 16-18 encode parallelism/independence, not just raw ordering')
elif outcomes.get('2a_same_depth') and not outcomes.get('2b_cross_branch'):
    print('→ Probe handles same-depth parallel edges (training match) but')
    print('  struggles with cross-branch edges (different depth, different geometry)')
elif not outcomes.get('2a_same_depth') and not outcomes.get('2b_cross_branch'):
    print('→ Probe 2 fails even on its closest training distribution')
    print('  Threshold, training data, or probe architecture needs review')
else:
    print('→ Unexpected pattern — check class balance and per-plan counts')


---
## Combined experiment

Both modes run on the same corrupted DAG:
40% hard-edge deletion + 2 spurious shortcut insertions.

**Mode 1** sees the incomparable pairs created by deletion and tries to recover the missing edges.

**Mode 2** sees all existing edges (real + spurious shortcuts) and tries to identify the shortcuts.

**Summary metric:** for each plan, the combined pipeline produces a modified DAG.
We report how many of the ground-truth changes (additions + removals) were correctly proposed.


In [27]:
rate_c =cfg['combined_deletion']
n_spur_c=cfg['combined_spurious']
rng_c=random.Random(SEED+2)

combined_rows=[]
for goal in sorted(single_plans):
    plan=parse_plan_json(goal,DATA_DIR)
    if not plan: continue
    orig_edges=plan['dag_edges']

    # Step 1: delete hard edges
    hard=[(a,b) for a,b in orig_edges
          if not has_alternative_path(orig_edges,a,b)]
    if not hard: continue
    n_rem=max(1,math.ceil(rate_c*len(hard)))
    deleted=set(map(tuple,rng_c.sample(hard,min(n_rem,len(hard)))))
    after_del=[e for e in orig_edges if tuple(e) not in deleted]

    # Step 2: add spurious shortcuts
    # Combined uses same-depth parallel spurious (Mode 2a) for best probe performance
    # For combined: use same-depth spurious from the remaining incomparable pairs
    tmp_plan={'steps':plan['steps'],'dag_edges':after_del,
              'incomparable':list(get_incompat_pairs(
                  list(plan['steps'].values()),after_del))}
    shortcuts=find_spurious_same_depth(tmp_plan)
    if not shortcuts:  # fallback to cross-branch if no same-depth available
        shortcuts=find_spurious_cross_branch(tmp_plan)
    added=rng_c.sample(shortcuts,min(n_spur_c,len(shortcuts))) if shortcuts else []
    corrupt_edges=after_del+list(added)

    # Build eval rows: incomparable pairs (Mode 1) + all edges (Mode 2)
    steps_list=list(plan['steps'].values())
    incompat=get_incompat_pairs(steps_list,corrupt_edges)
    for a,b in incompat:
        combined_rows.append({'goal':goal,'a':a,'b':b,'mode':'M1',
                              'y_true':1 if (a,b) in deleted else 0})
    for a,b in corrupt_edges:
        combined_rows.append({'goal':goal,'a':a,'b':b,'mode':'M2',
                              'y_true':1 if (a,b) in set(map(tuple,added)) else 0})

combined_df=pd.DataFrame(combined_rows)
m1_sub=combined_df[combined_df['mode']=='M1'].reset_index(drop=True)
m2_sub=combined_df[combined_df['mode']=='M2'].reset_index(drop=True)
print(f'Combined eval: M1 rows={len(m1_sub)} (pos={m1_sub.y_true.sum()})  '
      f'M2 rows={len(m2_sub)} (pos={m2_sub.y_true.sum()})')

# Evaluate
m1c_probe,_=sweep_probe(m1_sub,p1_score,   'Combined M1 probe')
m1c_llm,  _=sweep_llm_m1(m1_sub,           'Combined M1 LLM')
m2c_probe,_=sweep_probe(m2_sub,p2_spurious, 'Combined M2 probe')
m2c_llm   =sweep_llm_m2(m2_sub,             'Combined M2 LLM-inv')

print('\n=== COMBINED EXPERIMENT RESULTS ===')
for name,df in [('Mode 1 Probe 1',m1c_probe),('Mode 1 LLM direct',m1c_llm),
                ('Mode 2 Probe 2',m2c_probe),('Mode 2 LLM low-confidence',m2c_llm)]:
    b=best_f1(df)
    print(f'  {name:22}: F1={b.f1:.3f}  prec={b.precision:.3f}  '
          f'rec={b.recall:.3f}  t={b.threshold}')

In [28]:
# Mode 1
all_m1 = []
for rate, res in m1_results.items():
    for sys, df in res.items():
        df2=df.copy(); df2['rate']=rate; df2['system']=sys; all_m1.append(df2)
pd.concat(all_m1).to_csv(CONTENT/'enrichment_mode1.csv', index=False)

# Mode 2 — one CSV per subcondition
all_m2 = []
for (mode_tag,n), res in m2_results.items():
    for sys, df in res.items():
        df2=df.copy(); df2['mode']=mode_tag; df2['n_spurious']=n
        df2['system']=sys; all_m2.append(df2)
pd.concat(all_m2).to_csv(CONTENT/'enrichment_mode2.csv', index=False)

# Combined
for name,df in [('m1_probe',m1c_probe),('m1_llm',m1c_llm),
                ('m2_probe',m2c_probe),('m2_llm',m2c_llm)]:
    df.to_csv(CONTENT/f'combined_{name}.csv', index=False)

print('Saved: enrichment_mode1.csv  enrichment_mode2.csv')
print('       combined_m1/m2 probe/llm CSVs')
print('       mode1_scaling.png  mode2_scaling.png')
print()
print('=== FINAL SUMMARY ===')
for rate in cfg['deletion_rates']:
    f=best_f1(m1_results[rate]['probe']).f1
    lf=best_f1(m1_results[rate]['llm']).f1
    print(f'Mode 1 {int(rate*100)}% del: Probe={f:.3f}  LLM={lf:.3f}  +{f-lf:.3f}')
for mode_tag in ['2a_same_depth','2b_cross_branch']:
    for n in cfg['spurious_levels']:
        key=(mode_tag,n)
        if key not in m2_results: continue
        f=best_f1(m2_results[key]['probe']).f1
        lf=best_f1(m2_results[key]['llm']).f1
        print(f'Mode {mode_tag} n={n}:  Probe={f:.3f}  LLM={lf:.3f}  {f-lf:+.3f}')
